# SCAF Checkpoint Causal Leak Analysis
# Fock-PARFLM v2.1 — Comprehensive Post-Training Audit

Multi-phase causal analysis of a trained Fock-PARFLM checkpoint using the
**SCAF (SemSimula Causal Auditing Framework)** library.

## Phases

1. **Phase 1 — SCAF Full Audit**: Controls + future perturbation + target
   relocation + mediation attribution.
2. **Phase 2 — Interventional Leak Frame**: Build a tidy counterfactual
   dataset and compute exact ATE with paired inference.
3. **Phase 3 — Trained-Scale Leak Probe**: Legacy probe (future perturbation
   at trained scale + honest vs standard PPL).
4. **Phase 4 — Register Diagnostics**: Routing quality (entropy, diversity,
   alpha\_max) per layer.
5. **Phase 5 — Visualisation Dashboard**: 6-panel figure summarising all
   findings.
6. **Phase 6 — DoWhy/EconML Estimands** *(optional)*: Formal ATE, CATE by
   distance-to-cut, refutation tests.
7. **Phase 7 — Stiffness Audit** *(optional)*: reload a list of checkpoints
   from one run and score the Verlet stability bound (omega\*dt vs. 2) at
   each step, to test whether logged gradient spikes / watchdog reloads
   line up with the well curvature crossing that bound.

All results are persisted to a configurable GDrive location.


In [ ]:
# ── Cell 0: Configuration ──────────────────────────────────────────
# ── Checkpoint to analyse ──
CHECKPOINT_PATH = '/content/drive/MyDrive/semsimula_fock_multixi_structured_vtheta/A2/seed0/ckpt_best.pt'

# ── Output directory (GDrive) ──
OUTPUT_DIR = '/content/drive/MyDrive/scaf_analysis_results'

# ── Model family (auto-detected if checkpoint has 'model_cfg') ──
#    'auto'            → read from checkpoint['model_cfg'] /
#                        ['v_theta_variant'] / ['v_theta_kind']
#    'mlp'             → MLP V_theta (ScalarPotentialMultiXi)
#    'gaussian'        → Gaussian V_theta, ambiguous isotropic/anisotropic —
#                        resolved via GAUSSIAN_VARIANT below (default: aniso)
#    'gaussian_aniso'  → AnisotropicDepthConditionedGaussianVTheta (explicit)
#    'gaussian_iso'    → DepthConditionedMultiContextGaussianVTheta (explicit)
#    'sq3'             → MixtureQuadraticVTheta via StructuredVThetaMultiXiAdapter
MODEL_FAMILY = 'auto'

# ── Which Gaussian V_theta implementation to build when the detected/
#    configured family is the ambiguous 'gaussian' tag. Most checkpoints
#    worth debugging at d=384/d=768 scale use the anisotropic (low-rank
#    precision) variant, so it is the default. Set to 'iso' for older
#    isotropic-only checkpoints. Ignored when MODEL_FAMILY is already the
#    explicit 'gaussian_aniso' / 'gaussian_iso'.
GAUSSIAN_VARIANT = 'aniso'   # 'aniso' or 'iso'

# ── Anisotropic Gaussian V_theta: low-rank precision correction rank ──
#    Sigma_k^{-1} = diag(a_k) + B_k @ B_k^T,  B_k in R^{d x ANISO_RANK}.
#    Only used when the aniso variant is selected. Must match the rank the
#    checkpoint was trained with, or the state_dict load below will report
#    a B_proj shape mismatch.
ANISO_RANK = 4

# ── Override model config (only used if MODEL_FAMILY != 'auto') ──
D              = 256
L              = 8
N_REGISTERS    = 16
V_HIDDEN       = 1024
V_DEPTH        = 3
XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
BLOCK_SIZE     = 512
VOCAB_SIZE     = 50257
MAX_LEN        = 1024

# ── Gaussian V_theta overrides ──
V_THETA_N_HEADS        = 4
V_THETA_WELLS_PER_HEAD = 8

# ── SQ3 V_theta overrides ──
K_MIX    = 8
SQ3_TAU  = 1.0

# ── Integrator (only used when MODEL_FAMILY != 'auto') ──
#    A checkpoint carrying 'model_cfg' already records its integrator and it
#    is used verbatim; these are the fallback for older checkpoints that have
#    no config blob. Getting them wrong reconstructs a *different dynamical
#    system* from the one that was trained -- the weights still load, so the
#    error is silent -- and the audit then certifies the wrong model.
INTEGRATOR            = 'verlet'   # 'verlet' | 'baoab' | 'baoab_cfc'
VTHETA_ANALYTIC_FORCE = False      # closed-form -grad V_theta
LANGEVIN_T            = 0.0        # thermostat temperature (0 = deterministic)

# ── Data ──
DATASET = 'tinystories'   # 'tinystories' or 'openwebtext'
MAX_TRAIN_TOKENS = 5_000_000

# ── SCAF audit parameters ──
SCAF_SEQ_LEN     = 128
SCAF_N_SEQS      = 16
SCAF_N_TARGETS   = 64
SCAF_MICRO_BATCH = 4
SCAF_SEED        = 0

# ── Leak frame parameters ──
FRAME_N_SEQS     = 16
FRAME_N_PAIRS    = 4
FRAME_SPLITS     = (0.25, 0.5, 0.75)

# ── Legacy probe parameters ──
LEGACY_PROBE_N_PAIRS = 4
LEGACY_HONEST_K      = 256

# ── Register diagnostics ──
DIAG_BATCHES  = 10
DIAG_BATCH_SZ = 4

# ── DoWhy (Phase 6) ──
RUN_DOWHY = True

# ── Stiffness audit (Phase 7, optional) ──
#    Reloads each of these checkpoints (same run, different steps) and scores
#    the omega*dt distribution against the Verlet stability bound derived in
#    docs/BAOAB/PyTorch_Implementation_of_CfC_BAOAB_in_Fock-PARFLM.md sec.4.1.
#    Each path must be a checkpoint saved by one of the production OWT
#    notebooks (i.e. carries 'model_cfg' and 'step'); ckpt_best.pt alone is
#    not enough since it is one point, not a trajectory over steps. Leave
#    empty to skip Phase 7 entirely.
STIFFNESS_CKPT_PATHS = []   # e.g. ['/content/drive/.../ckpt_step6000.pt', ...]
STIFFNESS_N_BATCHES  = 4    # validation batches averaged into each report
#    Optional: step indices of logged watchdog reloads / gradient spikes for
#    this run (read them off the training_log.jsonl), drawn as vertical
#    reference lines on the omega*dt-vs-step plot for direct comparison.
WATCHDOG_RELOAD_STEPS = []   # e.g. [6200, 7450, 9100]

print(f'Checkpoint:  {CHECKPOINT_PATH}')
print(f'Output:      {OUTPUT_DIR}')
print(f'Model:       {MODEL_FAMILY}  (gaussian_variant={GAUSSIAN_VARIANT}, aniso_rank={ANISO_RANK})')
print(f'Dataset:     {DATASET}')
print(f'Integrator:  {INTEGRATOR} (manual path only; '
      f'analytic_vtheta={VTHETA_ANALYTIC_FORCE}, langevin_T={LANGEVIN_T})')
print(f'Stiffness audit: {len(STIFFNESS_CKPT_PATHS)} checkpoint(s) queued '
      f'(Phase 7{"" if STIFFNESS_CKPT_PATHS else " -- skipped, list is empty"})')


In [ ]:
# ── Cell 1: Environment + Drive Mount ─────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, copy
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
SCAF_URL    = 'https://github.com/dimitarpg13/semsimula-scaf.git'
REPO_BRANCH = 'main'
# Pinned, not 'main': the CfC/BAOAB compatibility fixes (well_parameters
# signature, capabilities(), InertIntervention, ...) live on this open PR and
# have not been merged. Move this back to 'main' once that PR lands, or the
# install below will silently fetch an older SCAF that predates the fixes.
SCAF_BRANCH = 'bug_fixes_and_cfc_baoab_comapt'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib seaborn')
    # PEP 508 direct-URL form, not '#egg=name[extras]': pip made the legacy
    # egg-fragment-with-extras syntax a hard error (pip >= ~25.0), and the
    # old form raises 'invalid-egg-fragment' rather than installing anything.
    _sh(f'pip install -q "semsimula-scaf[pywhy,plot] @ git+{SCAF_URL}@{SCAF_BRANCH}"')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

OUT_PATH = Path(OUTPUT_DIR)
OUT_PATH.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'scaleup/debug',
            'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'OUT_PATH  = {OUT_PATH}')


In [ ]:
# ── Cell 2: GPU + Imports ─────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')
else:
    print('WARNING: No GPU detected. Analysis will run on CPU (slow).')

import scaf
print(f'SCAF version: {scaf.__version__ if hasattr(scaf, "__version__") else "dev"}')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig

print('Imports OK')


In [ ]:
# ── Cell 3: Load Validation Data ──────────────────────────────────
from data_module import get_batch

if DATASET == 'tinystories':
    from data_module import load_tiny_stories
    train_ids, val_ids = load_tiny_stories(
        n_train_files=1, val_frac=0.01, max_train_tokens=MAX_TRAIN_TOKENS)
elif DATASET == 'openwebtext':
    from data_module import load_openwebtext
    train_ids, val_ids = load_openwebtext()
else:
    raise ValueError(f'Unknown dataset: {DATASET}')

print(f'Dataset: {DATASET}')
print(f'  train: {len(train_ids):,}   val: {len(val_ids):,}')

val_tokens = torch.from_numpy(val_ids[:SCAF_N_SEQS * SCAF_SEQ_LEN].astype(np.int64))
val_tokens = val_tokens.reshape(SCAF_N_SEQS, SCAF_SEQ_LEN)
print(f'  SCAF corpus: {val_tokens.shape}')


In [ ]:
# ── Cell 4: Reconstruct Model from Checkpoint ────────────────────
from dataclasses import asdict
import math as _math

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint loaded: {Path(CHECKPOINT_PATH).name}')
print(f'  Keys: {sorted(ckpt.keys())}')


def _detect_vtheta_family_from_state_dict(sd):
    """Infer which V_theta implementation a checkpoint was trained with,
    straight from its state_dict's parameter names/shapes.

    This is the ground truth for the 'from_cfg' path below: model_cfg
    (FockMultiXiPARFConfig) has no field recording that V_theta was
    swapped from the default MLP to a Gaussian family post-construction
    -- every production notebook does that swap the same way Cell 4's
    manual (non-from_cfg) branch does further down -- so a checkpoint
    with model_cfg gives no other way to tell MLP from Gaussian apart
    before load_state_dict silently drops every V_theta.* key as
    'unexpected' and leaves a random, untrained MLP in its place.

    Returns (family, info) with family in {'mlp', 'gaussian_iso',
    'gaussian_aniso', 'sq3', 'unknown'} and info holding the shape-derived
    hyperparameters (K, n_ctx, rank, d, depth_conditioned) needed to
    rebuild it.
    """
    vt_keys = [k for k in sd if k.startswith('V_theta.')]
    if not vt_keys:
        return 'unknown', {}

    has_bank = any('.bank.banks.' in k for k in vt_keys)
    has_direct_banks = any(k.startswith('V_theta.banks.') for k in vt_keys)
    has_B_proj = any('.B_proj.' in k for k in vt_keys)
    has_mu_a_proj = (any('.mu_proj.' in k for k in vt_keys)
                     and any('.a_proj.' in k for k in vt_keys))
    has_pi_proj = any('.pi_proj.' in k for k in vt_keys)
    has_depth_code = 'V_theta.depth_code' in vt_keys

    if has_pi_proj:
        return 'sq3', {}
    if has_mu_a_proj and (has_bank or has_direct_banks):
        family = 'gaussian_aniso' if has_B_proj else 'gaussian_iso'
        bank_prefix = 'V_theta.bank.banks.' if has_bank else 'V_theta.banks.'
        ctx_ids = sorted({
            int(k[len(bank_prefix):].split('.')[0]) for k in vt_keys
            if k.startswith(bank_prefix)
        })
        mu_w = sd[f'{bank_prefix}{ctx_ids[0]}.mu_proj.weight']
        d = mu_w.shape[1]
        K = mu_w.shape[0] // d
        info = dict(n_ctx=len(ctx_ids), K=K, d=d, bank_prefix=bank_prefix,
                    depth_conditioned=has_depth_code)
        if family == 'gaussian_aniso':
            B_w = sd[f'{bank_prefix}{ctx_ids[0]}.B_proj.weight']
            info['rank'] = B_w.shape[0] // (K * d)
        return family, info
    if any('.net.' in k for k in vt_keys):
        return 'mlp', {}
    return 'unknown', {}


# ── Determine model family ──
detected_family = MODEL_FAMILY
if MODEL_FAMILY == 'auto':
    if 'model_cfg' in ckpt:
        cfg_dict = ckpt['model_cfg']
        print(f'  model_cfg found in checkpoint (d={cfg_dict.get("d")}, L={cfg_dict.get("L")})')
        detected_family = 'from_cfg'
    elif 'v_theta_variant' in ckpt:
        detected_family = ckpt['v_theta_variant']
        print(f'  v_theta_variant from checkpoint: {detected_family}')
    elif 'v_theta_kind' in ckpt:
        detected_family = ckpt['v_theta_kind']
        print(f'  v_theta_kind from checkpoint: {detected_family}')
    else:
        detected_family = 'mlp'
        print(f'  No model_cfg in checkpoint — defaulting to: {detected_family}')

# ── Integrator settings for the manual path ──
# Passed as **kwargs so this notebook still runs against a checkout that
# predates the CfC/BAOAB propagator, as long as INTEGRATOR is left at
# 'verlet' (which is exactly what those older checkpoints used).
import dataclasses as _dc0
_cfg_fields = {f.name for f in _dc0.fields(FockMultiXiPARFConfig)}
_integrator_kwargs = {}
if 'integrator' in _cfg_fields:
    _integrator_kwargs = dict(
        integrator=INTEGRATOR,
        vtheta_analytic_force=VTHETA_ANALYTIC_FORCE,
        langevin_T=LANGEVIN_T,
    )
elif INTEGRATOR != 'verlet':
    raise RuntimeError(
        f'INTEGRATOR={INTEGRATOR!r} but this checkout of '
        'FockMultiXiPARFConfig has no integrator field. Pull '
        'semsimula-paper and restart the runtime.')

# ── Build model ──
if detected_family == 'from_cfg':
    cfg_dict = dict(ckpt['model_cfg'])
    logfreq_path = cfg_dict.get('logfreq_path', '')
    if logfreq_path and not Path(logfreq_path).exists():
        local_lf = CA_DIR / 'scaleup' / 'results' / Path(logfreq_path).name
        if local_lf.exists():
            cfg_dict['logfreq_path'] = str(local_lf)
        else:
            counts = np.bincount(train_ids.astype(np.int64),
                                 minlength=cfg_dict.get('vocab_size', VOCAB_SIZE)).astype(np.float64)
            p = (counts + 1.0) / (counts.sum() + cfg_dict.get('vocab_size', VOCAB_SIZE))
            surprisal = (-np.log(p)).astype(np.float32)
            _lf_tmp = Path('/tmp/logfreq_scaf.npy')
            np.save(str(_lf_tmp), surprisal)
            cfg_dict['logfreq_path'] = str(_lf_tmp)
            print(f'  Computed logfreq surprisal -> {_lf_tmp}')

    # A checkpoint trained after a config field was added cannot be rebuilt
    # by an older checkout. Say so plainly: the bare TypeError from the
    # dataclass names one field at a time and reads like a corrupt file.
    import dataclasses as _dc
    _known = {f.name for f in _dc.fields(FockMultiXiPARFConfig)}
    _unknown = sorted(set(cfg_dict) - _known)
    if _unknown:
        raise RuntimeError(
            f'Checkpoint config has fields this checkout does not know: '
            f'{_unknown}. The repo is older than the checkpoint -- pull '
            f'semsimula-paper and restart the runtime. (Fields such as '
            f"'integrator' / 'vtheta_analytic_force' / 'langevin_T' arrived "
            f'with the CfC/BAOAB propagator.)')

    model_cfg = FockMultiXiPARFConfig(**cfg_dict)
    model = FockMultiXiPARFLM(model_cfg)
    print(f'  Built model from checkpoint config (d={model_cfg.d}, L={model_cfg.L})')

    # model_cfg alone never distinguishes MLP V_theta from a Gaussian
    # family swapped in post-construction (see the helper's docstring
    # above) -- detect it from the checkpoint's own state_dict and swap
    # before load_state_dict, or every V_theta.* key silently becomes
    # 'unexpected' and the audit runs against an untrained random MLP.
    _vt_family, _vt_info = _detect_vtheta_family_from_state_dict(
        ckpt['model_state_dict'])
    print(f'  V_theta family (from state_dict): {_vt_family}  {_vt_info}')
    if _vt_family in ('gaussian_iso', 'gaussian_aniso'):
        _d, _K, _n_ctx = _vt_info['d'], _vt_info['K'], _vt_info['n_ctx']
        if _vt_family == 'gaussian_aniso':
            from model_aniso_gaussian_vtheta import (
                AnisotropicDepthConditionedGaussianVTheta,
                install_aniso_depth_routing)
            model.V_theta = AnisotropicDepthConditionedGaussianVTheta(
                d=_d, K=_K, n_ctx=_n_ctx, n_layers=model_cfg.L,
                rank=_vt_info['rank'], w_scale=1.0,
                init_log_precision=-_math.log(_d), precision_max=2.0 / _d,
                code_init_std=0.02,
            )
            install_aniso_depth_routing(model)
            print(f'  V_theta: Anisotropic Gaussian {_n_ctx}x{_K}, '
                  f'rank={_vt_info["rank"]} (swapped in from state_dict)')
        else:
            from model_gaussian_vtheta import (
                DepthConditionedMultiContextGaussianVTheta,
                install_depth_routing)
            model.V_theta = DepthConditionedMultiContextGaussianVTheta(
                d=_d, K=_K, n_ctx=_n_ctx, n_layers=model_cfg.L, w_scale=1.0,
                init_log_precision=-_math.log(_d), precision_max=2.0 / _d,
                code_init_std=0.02,
            )
            install_depth_routing(model)
            print(f'  V_theta: Isotropic Gaussian {_n_ctx}x{_K} '
                  '(swapped in from state_dict)')
    elif _vt_family == 'sq3':
        raise RuntimeError(
            "Checkpoint's V_theta state_dict looks like the SQ3 structured "
            "quadratic family, but Cell 4's 'from_cfg' path has no SQ3 "
            "swap-in (SQ3 checkpoints normally carry 'recipe', not "
            "'model_cfg' -- see Phase 7's _build_model_from_recipe_ckpt "
            "for that reconstruction path).")
    elif _vt_family == 'unknown':
        print('  WARNING: could not identify V_theta family from '
              "state_dict; leaving the default MLP V_theta. Check the "
              "'missing'/'unexpected' key lists below carefully.")
    # else: 'mlp' -- FockMultiXiPARFLM's default V_theta is already correct.

else:
    logfreq_path = CA_DIR / 'scaleup' / 'results'
    lf_file = logfreq_path / f'logfreq_surprisal_{DATASET}.npy'
    if not lf_file.exists():
        counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
        p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
        surprisal = (-np.log(p)).astype(np.float32)
        np.save(str(lf_file), surprisal)
        print(f'  Computed logfreq -> {lf_file}')

    model_cfg = FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN, L=L,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(lf_file),
        logfreq_init_alpha=0.1, init_gamma=1.0,
        fixed_gamma=ckpt.get('gamma', ckpt.get('fixed_gamma', 0.3)),
        causal_force=True, ln_after_step=True,
        xi_channels=XI_CHANNELS, xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_phi_hidden=128, v_phi_theta_hidden=128,
        top_k=8, v_phi_n_heads=1, score_head_hidden=32,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=True,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        use_output_bias=False, tie_embeddings=True,
        fock_version='v2', n_registers=N_REGISTERS,
        reverse_channel=True,
        d_k=64, tau_create_init=8.0, creation_gate_hidden=64,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
        stack_discipline=True, prefix_causal_registers=True,
        **_integrator_kwargs,
    )
    model = FockMultiXiPARFLM(model_cfg)

    if detected_family in ('gaussian', 'gaussian_aniso', 'gaussian_iso'):
        # Explicit checkpoint tags win outright; the bare 'gaussian' tag is
        # ambiguous (older checkpoints/metadata don't distinguish variants)
        # and falls back to the configured default (GAUSSIAN_VARIANT).
        if detected_family == 'gaussian_iso':
            gaussian_variant = 'iso'
        elif detected_family == 'gaussian_aniso':
            gaussian_variant = 'aniso'
        else:
            gaussian_variant = GAUSSIAN_VARIANT

        if gaussian_variant == 'aniso':
            from model_aniso_gaussian_vtheta import (
                AnisotropicDepthConditionedGaussianVTheta,
                install_aniso_depth_routing)
            model.V_theta = AnisotropicDepthConditionedGaussianVTheta(
                d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS, n_layers=L,
                rank=ANISO_RANK, w_scale=1.0, init_log_precision=-_math.log(D),
                precision_max=2.0 / D, code_init_std=0.02,
            )
            install_aniso_depth_routing(model)
            print(f'  V_theta: Anisotropic Gaussian {V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}, '
                  f'rank={ANISO_RANK}')
        else:
            from model_gaussian_vtheta import (
                DepthConditionedMultiContextGaussianVTheta, install_depth_routing)
            model.V_theta = DepthConditionedMultiContextGaussianVTheta(
                d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS, n_layers=L,
                w_scale=1.0, init_log_precision=-_math.log(D),
                precision_max=2.0 / D, code_init_std=0.02,
            )
            install_depth_routing(model)
            print(f'  V_theta: Isotropic Gaussian {V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')

    elif detected_family == 'sq3':
        from model_structured_vtheta import MixtureQuadraticVTheta
        xi_d = XI_CHANNELS * D
        inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
        torch.nn.Module.__init__(inner)
        inner.d = D; inner.K = K_MIX; inner.tau = SQ3_TAU
        inner.mu_proj = nn.Linear(xi_d, K_MIX * D)
        inner.a_proj  = nn.Linear(xi_d, K_MIX * D)
        inner.pi_proj = nn.Linear(xi_d, K_MIX)
        inner.b_proj  = nn.Linear(xi_d, 1)
        inner._init_weights(0.0)

        class _Adapter(nn.Module):
            def __init__(self, inner, K, d):
                super().__init__()
                self.inner = inner; self.K = K; self.d = d
            def forward(self, xis, h):
                return self.inner(xis.reshape(*xis.shape[:2], -1), h)

        model.V_theta = _Adapter(inner, K=XI_CHANNELS, d=D)
        print(f'  V_theta: SQ3 Mixture K_mix={K_MIX}')

    else:
        print(f'  V_theta: MLP (default)')

# ── Load weights ──
missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
if missing:
    print(f'  Missing keys:    {missing}')
if unexpected:
    print(f'  Unexpected keys: {unexpected}')

# Isotropic <-> anisotropic mismatch is the most common cause of missing/
# unexpected V_theta keys: 'B_proj' only exists on the anisotropic bank's
# low-rank precision factor. Surface this immediately rather than letting
# it manifest as silently-wrong wells downstream.
_b_proj_missing = any('B_proj' in k for k in missing)
_b_proj_unexpected = any('B_proj' in k for k in unexpected)
if _b_proj_missing:
    print('  WARNING: checkpoint has no B_proj weights but the model was '
          "built with the anisotropic V_theta. Set GAUSSIAN_VARIANT = 'iso' "
          '(or MODEL_FAMILY = \'gaussian_iso\') and re-run.')
elif _b_proj_unexpected:
    print("  WARNING: checkpoint has B_proj weights but the model was built "
          "with the isotropic V_theta. Set GAUSSIAN_VARIANT = 'aniso' "
          "(or MODEL_FAMILY = 'gaussian_aniso') and re-run.")

model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters())
n_vt = sum(p.numel() for p in model.V_theta.parameters())
print(f'  Total params:  {n_params:,}')
print(f'  V_theta params: {n_vt:,}')
print(f'  gamma: {model.gamma.item():.3f}')

# ── Dynamics identity ──
# Two runs of the same architecture under different integrators are different
# dynamical systems, so the audit record has to state which one it certified.
_integ = getattr(model_cfg, 'integrator', 'verlet')
_langT = getattr(model_cfg, 'langevin_T', 0.0)
print(f'  integrator: {_integ}'
      f'  analytic_vtheta={getattr(model_cfg, "vtheta_analytic_force", False)}'
      f'  langevin_T={_langT}')
if _integ != 'verlet':
    print('    (second-order state is the pair (h, v); the CfC substep '
          'propagates the harmonic part of V_theta in closed form)')
if _langT:
    print('    NOTE: thermostat is on. SCAF disables the eval-time noise for '
          'the duration of each probe, or every verdict would be noise.')

# ── Checkpoint metadata ──
ckpt_meta = {}
for k in ['step', 'val_ppl', 'best_val_ppl', 'val_loss', 'gamma',
          'variant', 'experiment', 'tag', 'seed', 'v_theta_kind',
          'v_theta_variant', 'K_mix']:
    if k in ckpt:
        ckpt_meta[k] = ckpt[k]
        print(f'  ckpt[{k}] = {ckpt[k]}')

del ckpt
gc.collect()
print('Model loaded OK')


## Phase 1 — SCAF Full Audit

Runs the complete SCAF audit battery:
- **Controls**: determinism, placebo, positive
- **Probes**: future perturbation ($\ell_\infty$ logit deviation), target relocation (honest PPL gap)
- **Diagnostics**: mediation attribution (if leak found)


In [ ]:
# ── Cell 5: Phase 1 — SCAF Full Audit ─────────────────────────────

print('=' * 64)
print('  PHASE 1: SCAF Full Audit')
print('=' * 64)

scorecard = scaf.audit(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=SCAF_N_SEQS,
    n_targets=SCAF_N_TARGETS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
    mediation=True,
)

print(scorecard.summary())

# A mediator SCAF could not actually reach is an adapter defect, not an
# exoneration: it is excluded from attribution rather than scored at zero.
_med = scorecard.get('mediation') if hasattr(scorecard, 'get') else None
if _med is not None and _med.detail.get('never_fired'):
    print(f'\nWARNING: mediators hooked but never reached: '
          f'{_med.detail["never_fired"]} -- their contribution is unmeasured, '
          f'not zero.')

sc_path = OUT_PATH / 'phase1_scaf_audit.json'
with open(str(sc_path), 'w') as f:
    json.dump(scorecard.to_dict(), f, indent=2, default=str)
print(f'Saved: {sc_path}')


## Phase 1.5 — Geometric Leak Probes (Tier A / Tier B)

Runs the SCAF **geometric** diagnostics directly against the loaded
checkpoint. These operate in the model's internal hidden-state geometry
rather than on logits, and run as *diagnostics* — they add a second,
structural axis of evidence on top of Phase 1's verdict without changing it.

- **Tier A — `HiddenStateLeakProbe`**: per-layer cosine deviation of hidden
  states under `do(future)`. Catches *latent* leaks that have corrupted the
  hidden state but haven't reached the logits yet (`latent_leak=True`).
- **Tier B — `BasinMembershipProbe`**: whether the future perturbation flips
  a past hidden state's dominant attractor well in the Gaussian $V_\theta$
  landscape — a discrete, structurally more severe signal than continuous
  cosine deviation.

Availability depends on the checkpoint's `V_theta` family:

| `V_theta` family | `has_hidden_states` | `has_vtheta_wells` |
|---|:---:|:---:|
| MLP (`ScalarPotentialMultiXi`) | ✅ | ❌ (skips loudly) |
| Isotropic Gaussian (`MixtureGaussianVTheta`) | ✅ | ✅ |
| Anisotropic Gaussian (`Anisotropic*GaussianVTheta`) | ✅ | ✅ |

Both isotropic and anisotropic Gaussian families work with Tier B: the
adapter normalises the isotropic 3-tuple `_components()` return
(`mu, a, w`) to a rank-0 low-rank factor, which is mathematically identical
to having no low-rank correction at all.

Both tiers score each arm in **its own** $V_\theta$ landscape. The wells are
functions of the context vectors $\xi$, which the layer step derives from the
hidden state entering that layer — so they differ per layer and per position,
and a future perturbation moves the landscape as well as the point sitting in
it. Holding the wells at their factual values would answer a different
question and would miss a leak that travels through $\xi$.

Model reconstruction (Cell 4) defaults to the **anisotropic** variant
(`GAUSSIAN_VARIANT = 'aniso'`) whenever the family resolves to the ambiguous
`'gaussian'` tag, since that's the most likely family for d384/d768
checkpoints. Set `GAUSSIAN_VARIANT = 'iso'` (or `MODEL_FAMILY =
'gaussian_iso'` / `'gaussian_aniso'` explicitly) if reconstruction picks the
wrong one — a `B_proj`-related missing/unexpected-key warning at load time
is the tell.

In [ ]:
# ── Cell 5b: Phase 1.5 — Geometric Leak Probes (Tier A/B) ────────

print('=' * 64)
print('  PHASE 1.5: Geometric Leak Probes (Tier A/B)')
print('=' * 64)

corpus_geo = scaf.TokenCorpus(val_tokens, seq_len=SCAF_SEQ_LEN, seed=SCAF_SEED)

with scaf.InterventableModel(model, device=DEVICE, dtype='float32') as im:
    print(f'  adapter:           {im.adapter.name}')
    print(f'  has_hidden_states: {im.caps.has_hidden_states}')
    print(f'  has_vtheta_wells:  {im.caps.has_vtheta_wells}')

    tier_a = None
    if im.caps.has_hidden_states:
        tier_a = scaf.HiddenStateLeakProbe(
            splits=FRAME_SPLITS, n_seqs=SCAF_N_SEQS, n_pairs=2,
            micro_batch=SCAF_MICRO_BATCH,
        ).run(im, corpus_geo)
        print(f'\n  [Tier A] {tier_a}')
        if not tier_a.skipped:
            per_layer_a = tier_a.detail['per_layer_delta_cos']
            print(f'    per-layer dcos: {[f"{v:.4e}" for v in per_layer_a]}')
            print(f'    peak layer:     {tier_a.detail["peak_layer"]}')
            print(f'    latent_leak:    {tier_a.detail["latent_leak"]}')
    else:
        print('\n  [Tier A] SKIPPED -- adapter does not expose hidden states')

    tier_b = None
    if im.caps.has_vtheta_wells:
        tier_b = scaf.BasinMembershipProbe(
            splits=FRAME_SPLITS, n_seqs=SCAF_N_SEQS, n_pairs=2,
            micro_batch=SCAF_MICRO_BATCH,
        ).run(im, corpus_geo)
        print(f'\n  [Tier B] {tier_b}')
        if not tier_b.skipped:
            per_layer_b = tier_b.detail['per_layer_crossing_rate']
            print(f'    per-layer beta: {[f"{v:.4e}" for v in per_layer_b]}')
            print(f'    worst layer:    {tier_b.detail["worst_layer"]}')
    else:
        print('\n  [Tier B] SKIPPED -- V_theta does not expose well '
              'parameters (expected for the MLP V_theta family)')

    geo_result = {
        'has_hidden_states': bool(im.caps.has_hidden_states),
        'has_vtheta_wells': bool(im.caps.has_vtheta_wells),
        'tier_a': tier_a.to_dict() if tier_a is not None else None,
        'tier_b': tier_b.to_dict() if tier_b is not None else None,
    }

geo_path = OUT_PATH / 'phase1_5_geometric_probes.json'
with open(str(geo_path), 'w') as f:
    json.dump(geo_result, f, indent=2, default=str)
print(f'\nSaved: {geo_path}')

## Phase 2 — Interventional Leak Frame

Builds a tidy counterfactual dataset where each row is a scored target
position under factual or $\text{do}(\text{future} := \text{resampled})$.
The ATE is computed with exact paired sign-flip inference.


In [ ]:
# ── Cell 6: Phase 2 — Interventional Leak Frame ──────────────────

print('=' * 64)
print('  PHASE 2: Interventional Leak Frame')
print('=' * 64)

frame = scaf.build_leak_frame(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=FRAME_N_SEQS,
    n_pairs=FRAME_N_PAIRS,
    splits=FRAME_SPLITS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
)

print(frame.summary(test=True))

ate_result = frame.ate_test()
print(f'\nATE = {ate_result["ate"]:+.6f} nats')
print(f'  p-value = {ate_result["p_value"]:.4e}')
print(f'  95% CI  = [{ate_result["ci"][0]:+.6f}, {ate_result["ci"][1]:+.6f}]')
print(f'  n_units = {ate_result["n_units"]}')

profile = frame.ate_by('distance_to_cut')
print(f'\nLeak profile by distance_to_cut:')
for dist, ate in sorted(profile.items()):
    print(f'  distance={dist:3d}  ATE={ate:+.6f} nats')

frame_path = OUT_PATH / 'phase2_leak_frame.csv'
frame.to_csv(str(frame_path))
print(f'\nSaved: {frame_path}')

ate_path = OUT_PATH / 'phase2_ate_result.json'
with open(str(ate_path), 'w') as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v)
               for k, v in ate_result.items()}, f, indent=2, default=str)
print(f'Saved: {ate_path}')


## Phase 2.5 — Geometric Leak Frame (CATE by layer / basin)

Builds a **separate** counterfactual dataset with `include_hidden_states=True`
and `include_basin_membership=True`. Each row is now keyed by
`(layer, position)` rather than `position` alone, which is why this is kept
out of Phase 2's exact-ATE frame — that frame's paired sign-flip test assumes
one row per scored position, and multiplying by layer count would understate
the standard error (pseudo-replication). Use this frame only for the
layer/basin breakdown, not as a substitute for Phase 2's headline ATE.

Kept intentionally small (`n_seqs`, `n_pairs` capped below the Phase 2
defaults) — this frame's row count scales with `n_layers`, which gets
expensive fast.

In [ ]:
# ── Cell 6b: Phase 2.5 — Geometric Leak Frame ────────────────────

print('=' * 64)
print('  PHASE 2.5: Geometric Leak Frame (CATE by layer / basin)')
print('=' * 64)

GEO_FRAME_N_SEQS  = min(FRAME_N_SEQS, 8)
GEO_FRAME_N_PAIRS = min(FRAME_N_PAIRS, 2)

geo_frame = scaf.build_leak_frame(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=GEO_FRAME_N_SEQS,
    n_pairs=GEO_FRAME_N_PAIRS,
    splits=FRAME_SPLITS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
    include_hidden_states=True,
    include_basin_membership=True,
)

print(geo_frame.summary(test=False))
print(f'  columns: {list(geo_frame.columns.keys())}')

layer_col = np.array(geo_frame.column('layer'))
arm_col = np.array(geo_frame.column('future_perturbed'))
cf_mask = arm_col == 1  # deviation is 0 by construction on the factual arm

print('\n  Mean cosine deviation by layer (counterfactual arm):')
dev_col = np.array(geo_frame.column('hidden_cos_dev'))
for ell in sorted(set(layer_col.tolist())):
    m = cf_mask & (layer_col == ell)
    if m.any():
        print(f'    layer {ell:2d}: {dev_col[m].mean():.4e}')

if 'basin_changed' in geo_frame.columns:
    print('\n  Basin-crossing rate by layer (counterfactual arm):')
    basin_col = np.array(geo_frame.column('basin_changed'))
    for ell in sorted(set(layer_col.tolist())):
        m = cf_mask & (layer_col == ell)
        if m.any():
            print(f'    layer {ell:2d}: {basin_col[m].mean():.4e}')
else:
    print('\n  basin_changed column absent -- checkpoint V_theta has no '
        'well parameters (expected for the MLP V_theta family)')

geo_frame_path = OUT_PATH / 'phase2_5_geometric_leak_frame.csv'
geo_frame.to_csv(str(geo_frame_path))
print(f'\nSaved: {geo_frame_path}')

## Phase 3 — Legacy Trained-Scale Leak Probe

The original two-part probe from the training notebooks:
1. **Future perturbation** at trained scale (max logit deviation, mean dNLL)
2. **Honest vs standard PPL** on the same target tokens


In [ ]:
# ── Cell 7: Phase 3 — Legacy Trained Leak Probe ──────────────────
from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

print('=' * 64)
print('  PHASE 3: Legacy Trained-Scale Leak Probe')
print('=' * 64)

probe_res = probe_trained_leak(
    model, val_ids, device=DEVICE, context=BLOCK_SIZE,
    n_pairs=LEGACY_PROBE_N_PAIRS, use_float64=False)

print(f'\n  max|dlogit|(past) = {probe_res["max_dlogit_past"]:.3e}')
print(f'  mean dNLL(past)   = {probe_res["mean_dnll_past"]:+.6f} nats')
print(f'  gate zero control = {probe_res["gate_zero_control"]:.3e}')

honest_res = honest_ppl_test(
    model, val_ids, k=LEGACY_HONEST_K,
    context=BLOCK_SIZE, batch=DIAG_BATCH_SZ, device=DEVICE)

print(f'\n  Standard PPL (mid-window): {honest_res["ppl_mid_window"]:.2f}')
print(f'  Honest PPL (last-pos):     {honest_res["ppl_last_pos"]:.2f}')
print(f'  Paired diff:               {honest_res["paired_diff_nats"]:+.6f} '
      f'+/- {honest_res["paired_diff_se"]:.6f} nats')

leak_status = 'CLEAN' if honest_res['paired_diff_nats'] < 0.1 else 'LEAK'
print(f'  Verdict: [{leak_status}]')

model.train()
model.eval()

legacy_result = {
    'probe': probe_res,
    'honest_ppl': honest_res,
    'verdict': leak_status,
}
legacy_path = OUT_PATH / 'phase3_legacy_probe.json'
with open(str(legacy_path), 'w') as f:
    json.dump(legacy_result, f, indent=2, default=str)
print(f'\nSaved: {legacy_path}')


## Phase 4 — Register Diagnostics

Per-layer routing quality, read through `FockMultiXiPARFLM.set_fock_capture`
(the official zero-cost-when-off diagnostic).  Older Fock stacks exposed
`model.fock_layers[ell]._last_creation_alpha`; v2/v2.1 has a *shared*
`creation_gate_qkv` and no `fock_layers` attribute.

- **Normalised attention entropy** — `create_entropy / log(T)`; 0 = peaked, 1 = mean-pool
- **Register content diversity** — `1 - reg_cos_sim`; 0 = identical, 1 = orthogonal
- **$\alpha_{\max}$** — max creation-gate attention weight


In [ ]:
# ── Cell 8: Phase 4 — Register Diagnostics ───────────────────────
#
# FockMultiXiPARFLM (v2 / v2.1) has no `fock_layers` ModuleList.  Creation
# is a shared `creation_gate_qkv`; per-layer health is recorded by
# `set_fock_capture(True)` into `model._fock_capture`.  Same API the
# production OWT notebooks already use for the structural-health probe.

print('=' * 64)
print('  PHASE 4: Register Diagnostics')
print('=' * 64)

if not hasattr(model, 'set_fock_capture'):
    raise RuntimeError(
        f'{type(model).__name__} has no set_fock_capture().  Phase 4 needs '
        'FockMultiXiPARFLM (or a Fock stack that implements the capture API).'
    )

model.eval()
model.set_fock_capture(True)

entropy_per_layer = []
diversity_per_layer = []
alpha_max_per_layer = []
log_T = float(np.log(max(BLOCK_SIZE, 2)))

rng_diag = np.random.default_rng(42)

for bi in range(DIAG_BATCHES):
    x_np, _ = get_batch(val_ids, DIAG_BATCH_SZ, BLOCK_SIZE, rng_diag)
    x = torch.from_numpy(x_np).to(DEVICE)

    # The PARF/SARF force still needs autograd.grad at every layer, even
    # in eval.  Capture is a side-effect on model._fock_capture.
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    cap = list(model._fock_capture or [])
    model._fock_capture = []  # keep capture_stats on; just drain this batch

    if not cap:
        print(f'[WARN] batch {bi}: _fock_capture empty after forward '
              f'(fock_version={getattr(getattr(model, "cfg", None), "fock_version", "?")})')

    for rec in cap:
        ell = int(rec.get('layer', 0))
        ent_nats = rec.get('create_entropy')
        entropy_val = (float(ent_nats) / log_T) if ent_nats is not None else float('nan')
        amax = rec.get('create_alpha_max', float('nan'))
        amax = float(amax) if amax is not None else float('nan')
        cos_sim = rec.get('reg_cos_sim')
        div_val = (1.0 - float(cos_sim)) if cos_sim is not None else float('nan')

        while len(entropy_per_layer) <= ell:
            entropy_per_layer.append([])
            diversity_per_layer.append([])
            alpha_max_per_layer.append([])

        entropy_per_layer[ell].append(entropy_val)
        diversity_per_layer[ell].append(div_val)
        alpha_max_per_layer[ell].append(amax)

    del x, h0, h_L
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model.set_fock_capture(False)

n_layers_diag = len(entropy_per_layer)
diag_summary = {}
print(f'\n{"Layer":>6}  {"Entropy":>8}  {"alpha_max":>9}  {"Diversity":>9}')
print('-' * 40)
for ell in range(n_layers_diag):
    e = np.mean(entropy_per_layer[ell]) if entropy_per_layer[ell] else float('nan')
    a = np.mean(alpha_max_per_layer[ell]) if alpha_max_per_layer[ell] else float('nan')
    d = np.mean(diversity_per_layer[ell]) if diversity_per_layer[ell] else float('nan')
    diag_summary[ell] = {'entropy': e, 'alpha_max': a, 'diversity': d}
    print(f'{ell:6d}  {e:8.4f}  {a:9.4f}  {d:9.4f}')

mean_entropy = np.nanmean([v['entropy'] for v in diag_summary.values()])
mean_amax = np.nanmean([v['alpha_max'] for v in diag_summary.values()])
mean_div = np.nanmean([v['diversity'] for v in diag_summary.values()])
print(f'\nOverall: entropy={mean_entropy:.4f}  alpha_max={mean_amax:.4f}  diversity={mean_div:.4f}')

if mean_div >= 0.6 and 0.1 <= mean_entropy <= 0.5:
    routing_verdict = 'ROUTING'
elif mean_div < 0.3 and mean_entropy > 0.8:
    routing_verdict = 'MEAN-POOL'
else:
    routing_verdict = 'MIXED'
print(f'VERDICT: {routing_verdict}')

diag_path = OUT_PATH / 'phase4_register_diagnostics.json'
with open(str(diag_path), 'w') as f:
    json.dump({'per_layer': {str(k): v for k, v in diag_summary.items()},
               'overall': {'entropy': mean_entropy, 'alpha_max': mean_amax,
                           'diversity': mean_div, 'verdict': routing_verdict}},
              f, indent=2)
print(f'Saved: {diag_path}')


## Phase 5 — Visualisation Dashboard

Six-panel summary figure:
1. SCAF audit scorecard (text)
2. Within-window NLL profile (bar chart)
3. Leak profile by distance-to-cut (CATE)
4. Per-layer register entropy (bar)
5. Per-layer register diversity (bar)
6. Per-layer alpha\_max (bar)


In [ ]:
# ── Cell 9: Phase 5 — Visualisation Dashboard ────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3)

# ── Panel 1: Audit Scorecard (text) ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')

verdict = scorecard.verdict
verdict_color = {'CLEAN': '#2ecc71', 'LEAK': '#e74c3c', 'INVALID': '#f39c12'}[verdict]

summary_lines = []
summary_lines.append(f'SCAF Audit Verdict: {verdict}')
summary_lines.append('')
for cr in scorecard.controls:
    summary_lines.append(f'  {cr.status:4s}  {cr.name}: {cr.statistic:.3e} {cr.unit}')
for pr in scorecard.probes:
    summary_lines.append(f'  {pr.status:4s}  {pr.name}: {pr.statistic:.3e} {pr.unit}')
if scorecard.diagnostics:
    summary_lines.append('')
    for dr in scorecard.diagnostics:
        summary_lines.append(f'  {dr.status:4s}  {dr.name}: {dr.statistic:.3f} {dr.unit}')

tr = scorecard.get('target_relocation')
if tr and tr.detail:
    summary_lines.append('')
    summary_lines.append(f'  Standard PPL: {tr.detail.get("ppl_standard", "?"):.2f}')
    summary_lines.append(f'  Honest PPL:   {tr.detail.get("ppl_honest", "?"):.2f}')

summary_text = '\n'.join(summary_lines)
ax1.text(0.05, 0.95, summary_text, transform=ax1.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=verdict_color, alpha=0.15))
ax1.set_title('SCAF Audit Scorecard', fontsize=12, fontweight='bold')

# ── Panel 2: Within-Window NLL Profile ──
ax2 = fig.add_subplot(gs[0, 1])
if 'position_profile' in honest_res and honest_res['position_profile']:
    profile_data = honest_res['position_profile']
    positions = list(range(len(profile_data)))
    ax2.bar(positions, profile_data, color='steelblue', alpha=0.7, width=1.0)
    ax2.set_xlabel('Position in window')
    ax2.set_ylabel('Mean NLL (nats)')
    ax2.set_title('Within-Window NLL Profile', fontsize=12, fontweight='bold')
    ax2.axhline(y=np.mean(profile_data), color='red', linestyle='--', alpha=0.5, label='mean')
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'No position profile available', transform=ax2.transAxes,
             ha='center', va='center', fontsize=12)
    ax2.set_title('Within-Window NLL Profile', fontsize=12, fontweight='bold')

# ── Panel 3: Leak Profile by Distance to Cut ──
ax3 = fig.add_subplot(gs[1, 0])
if profile:
    dists = sorted(profile.keys())
    ates = [profile[d] for d in dists]
    colors = ['#e74c3c' if a > 0.01 else '#2ecc71' for a in ates]
    ax3.bar(dists, ates, color=colors, alpha=0.7)
    ax3.axhline(y=0, color='black', linewidth=0.5)
    ax3.set_xlabel('Distance to cut (positions)')
    ax3.set_ylabel('ATE (nats)')
    ax3.set_title('Leak Profile: CATE by Distance to Cut', fontsize=12, fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No leak frame data', transform=ax3.transAxes,
             ha='center', va='center', fontsize=12)
    ax3.set_title('Leak Profile', fontsize=12, fontweight='bold')

# ── Panel 4: Per-Layer Entropy ──
ax4 = fig.add_subplot(gs[1, 1])
if diag_summary:
    layers = sorted(diag_summary.keys())
    entropies = [diag_summary[l]['entropy'] for l in layers]
    ax4.bar(layers, entropies, color='#3498db', alpha=0.7)
    ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='routing threshold')
    ax4.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='mean-pool threshold')
    ax4.set_xlabel('Layer')
    ax4.set_ylabel('Normalised Entropy')
    ax4.set_title('Register Attention Entropy', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=8)
    ax4.set_ylim(0, 1.1)

# ── Panel 5: Per-Layer Diversity ──
ax5 = fig.add_subplot(gs[2, 0])
if diag_summary:
    diversities = [diag_summary[l]['diversity'] for l in layers]
    ax5.bar(layers, diversities, color='#27ae60', alpha=0.7)
    ax5.axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='routing threshold')
    ax5.set_xlabel('Layer')
    ax5.set_ylabel('Content Diversity')
    ax5.set_title('Register Content Diversity', fontsize=12, fontweight='bold')
    ax5.legend(fontsize=8)
    ax5.set_ylim(0, 1.1)

# ── Panel 6: Per-Layer alpha_max ──
ax6 = fig.add_subplot(gs[2, 1])
if diag_summary:
    amaxes = [diag_summary[l]['alpha_max'] for l in layers]
    ax6.bar(layers, amaxes, color='#e67e22', alpha=0.7)
    ax6.set_xlabel('Layer')
    ax6.set_ylabel('Mean alpha_max')
    ax6.set_title('Creation Gate Attention Strength', fontsize=12, fontweight='bold')
    ax6.set_ylim(0, 1.1)

# ── Save ──
fig.suptitle(f'SCAF Causal Leak Analysis — Fock v2.1 PARFLM\n'
             f'Checkpoint: {Path(CHECKPOINT_PATH).name}  |  Verdict: {verdict}',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()

dashboard_path = OUT_PATH / 'phase5_dashboard.png'
fig.savefig(str(dashboard_path), dpi=150, bbox_inches='tight')
print(f'Saved: {dashboard_path}')
plt.show()

# ── Additional: ATE bar chart ──
fig2, ax_ate = plt.subplots(figsize=(8, 4))
ate_val = ate_result['ate']
ci_lo, ci_hi = ate_result['ci']
color = '#2ecc71' if abs(ate_val) < 0.01 else '#e74c3c'
ax_ate.bar(['ATE'], [ate_val], color=color, alpha=0.7)
ax_ate.errorbar(['ATE'], [ate_val], yerr=[[ate_val - ci_lo], [ci_hi - ate_val]],
                fmt='none', color='black', capsize=10, linewidth=2)
ax_ate.axhline(y=0, color='black', linewidth=0.5)
ax_ate.set_ylabel('Effect (nats)')
ax_ate.set_title(f'Average Treatment Effect: do(future) on NLL\n'
                 f'ATE = {ate_val:+.6f} nats  (p = {ate_result["p_value"]:.4e})',
                 fontsize=11)
fig2.tight_layout()
ate_plot_path = OUT_PATH / 'phase5_ate_bar.png'
fig2.savefig(str(ate_plot_path), dpi=150)
print(f'Saved: {ate_plot_path}')
plt.show()


## Phase 6 — DoWhy/EconML Estimands *(optional)*

Formal causal estimation with:
- ATE via paired sign-flip test + bootstrap CI
- CATE by distance-to-cut via CausalForestDML
- Refutation tests (placebo treatment, random common cause)


In [ ]:
# ── Cell 10: Phase 6 — DoWhy/EconML Estimands ────────────────────

if not RUN_DOWHY:
    print('Phase 6 skipped (RUN_DOWHY = False)')
else:
    print('=' * 64)
    print('  PHASE 6: DoWhy/EconML Estimands')
    print('=' * 64)

    try:
        est_report = scaf.estimate_leak(
            frame,
            outcome='nll_within',
            cate_axes=('distance_to_cut',),
            cate_model='forest',
            refuters=('placebo_treatment_refuter', 'random_common_cause'),
            num_simulations=20,
            dowhy_inference=False,
        )
        print(est_report.summary())

        est_path = OUT_PATH / 'phase6_estimation_report.json'
        with open(str(est_path), 'w') as f:
            report_dict = {
                'ate': est_report.ate,
                'stderr': est_report.stderr,
                'ci': list(est_report.ci) if est_report.ci else None,
                'p_value': est_report.p_value,
                'reference_ate': est_report.reference_ate,
                'agrees_with_reference': est_report.agrees_with_reference,
                'refutations_ok': est_report.refutations_ok,
            }
            json.dump(report_dict, f, indent=2, default=str)
        print(f'Saved: {est_path}')

        # ── CATE plot ──
        if est_report.cate and 'distance_to_cut' in est_report.cate:
            cate_data = est_report.cate['distance_to_cut']
            fig3, ax_cate = plt.subplots(figsize=(10, 4))

            exact_profile = frame.ate_by('distance_to_cut')
            if exact_profile:
                dists = sorted(exact_profile.keys())
                exact_ates = [exact_profile[d] for d in dists]
                ax_cate.bar(dists, exact_ates, color='steelblue', alpha=0.4, label='Exact stratified')

            # cate_data is {'exact': [(dist, ate, n), ...], 'fit': {'model':
            # ..., 'points': [(dist, tau), ...], 'ate': ...}} (see
            # EstimationReport.cate in scaf/estimate/pywhy.py). The forest curve
            # lives under ['fit']['points'], not at the top level of cate_data --
            # iterating cate_data.keys() directly yields the strings 'exact' and
            # 'fit' rather than distances, which is what raised the inhomogeneous-
            # array error here.
            fit = (cate_data or {}).get('fit') or {}
            points = fit.get('points') or []
            if points:
                points = sorted(points, key=lambda p: p[0])
                forest_dists = [p[0] for p in points]
                forest_ates = [p[1] for p in points]
                ax_cate.plot(forest_dists, forest_ates, 'ro-', markersize=5,
                            label=f"{fit.get('model', 'CausalForestDML')}", linewidth=2)

            ax_cate.axhline(y=0, color='black', linewidth=0.5)
            ax_cate.set_xlabel('Distance to cut (positions)')
            ax_cate.set_ylabel('CATE (nats)')
            ax_cate.set_title('Heterogeneous Treatment Effect by Distance to Cut')
            ax_cate.legend()
            fig3.tight_layout()

            cate_path = OUT_PATH / 'phase6_cate_by_distance.png'
            fig3.savefig(str(cate_path), dpi=150)
            print(f'Saved: {cate_path}')
            plt.show()

    except ImportError as e:
        print(f'DoWhy/EconML not available: {e}')
        print('Install with: pip install "semsimula-scaf[pywhy]"')
    except Exception as e:
        print(f'Phase 6 error: {e}')
        import traceback
        traceback.print_exc()


In [ ]:
# ── Cell 11: Final Summary ────────────────────────────────────────

print()
print('=' * 64)
print('  SCAF CHECKPOINT ANALYSIS — FINAL SUMMARY')
print('=' * 64)
print()
print(f'Checkpoint: {Path(CHECKPOINT_PATH).name}')
print(f'Model:      Fock v2.1 PARFLM  (d={model_cfg.d}, L={model_cfg.L}, M={model_cfg.n_registers})')
print(f'Dynamics:   integrator={_integ}  langevin_T={_langT}')
print(f'Device:     {DEVICE}')
print()

print('Phase 1 — SCAF Audit')
print(f'  Verdict: {scorecard.verdict}')
for cr in scorecard.controls:
    print(f'    {cr.status:4s}  {cr.name}: {cr.statistic:.3e}')
for pr in scorecard.probes:
    print(f'    {pr.status:4s}  {pr.name}: {pr.statistic:.3e} {pr.unit}')
print()

print('Phase 2 — Leak Frame')
print(f'  ATE = {ate_result["ate"]:+.6f} nats  (p = {ate_result["p_value"]:.4e})')
print()

print('Phase 3 — Legacy Probe')
print(f'  max|dlogit|(past) = {probe_res["max_dlogit_past"]:.3e}')
print(f'  Honest PPL: {honest_res["ppl_last_pos"]:.2f}  '
      f'Standard PPL: {honest_res["ppl_mid_window"]:.2f}  '
      f'diff: {honest_res["paired_diff_nats"]:+.4f} nats  [{leak_status}]')
print()

print('Phase 4 — Register Diagnostics')
print(f'  Entropy: {mean_entropy:.4f}  Diversity: {mean_div:.4f}  '
      f'alpha_max: {mean_amax:.4f}  [{routing_verdict}]')
print()

print('Phase 1.5 — Geometric Leak Probes (Tier A/B)')
if geo_result['tier_a'] is not None:
    ta = geo_result['tier_a']
    print(f'  [Tier A] max_cos_dev={ta["statistic"]:.4e}  '
          f'peak_layer={ta["detail_peak_layer"]}  '
          f'latent_leak={ta["detail_latent_leak"]}  [{ta["status"]}]')
else:
    print('  [Tier A] SKIPPED (no hidden-state support)')
if geo_result['tier_b'] is not None:
    tb = geo_result['tier_b']
    print(f'  [Tier B] basin_crossing_rate={tb["statistic"]:.4e}  '
          f'worst_layer={tb["detail_worst_layer"]}  [{tb["status"]}]')
else:
    print('  [Tier B] SKIPPED (no V_theta well parameters)')
print()

print(f'All outputs saved to: {OUT_PATH}')
saved_files = sorted(OUT_PATH.glob('phase*'))
for f in saved_files:
    print(f'  {f.name}')
print()

overall_clean = (scorecard.verdict == 'CLEAN'
                 and leak_status == 'CLEAN'
                 and abs(ate_result['ate']) < 0.01)

if overall_clean:
    print('OVERALL ASSESSMENT: Model is CAUSALLY CLEAN')
    print('  No evidence of future information leakage across all analysis phases.')
else:
    print('OVERALL ASSESSMENT: FURTHER INVESTIGATION NEEDED')
    if scorecard.verdict != 'CLEAN':
        print(f'  SCAF audit: {scorecard.verdict}')
    if leak_status != 'CLEAN':
        print(f'  Legacy probe: {leak_status}')
    if abs(ate_result['ate']) >= 0.01:
        print(f'  ATE: {ate_result["ate"]:+.6f} nats')

print()
print('Done!')


## Phase 7 — Stiffness Audit *(optional)*

Retrospective test of the Verlet stability bound derived in
[PyTorch Implementation of CfC/BAOAB, §4.1](https://github.com/dimitarpg13/semantic_simulation/blob/main/docs/BAOAB/PyTorch_Implementation_of_CfC_BAOAB_in_Fock-PARFLM.md#41-the-verlet-stability-bound):
the explicit layer step is stable only while $\omega\Delta t < 2$, where
$\omega = \sqrt{K/\mathfrak{m}}$ and $K$ is the local curvature of the well
a token sits in. §4.2 of that same document flags this as untested against
real training data. This phase reloads a **list** of checkpoints from one
run (ideally bracketing a logged watchdog reload) and scores the
$\omega\Delta t$ distribution at each one, so the mechanism behind a spike
can be checked directly against the weights at that step instead of only
inferred from the gradient-norm log.

Requirements:

- the checkpoint carries either an embedded `model_cfg`
  (`ckpt['model_cfg']`, `ckpt['step']` — the schema the production OWT
  notebooks save) **or** a `recipe` dict (`ckpt['recipe']`, `ckpt['cell']`
  — the schema `colab_fock_multixi_structured_vtheta.ipynb` saves). This
  cell tries the former first and falls back to the latter automatically
  — no config needed from `MODEL_FAMILY`/`K_MIX`/`SQ3_TAU` in Cell 0.
- `model.V_theta` implements `harmonic_terms()` — true for the isotropic
  and anisotropic Gaussian families (`model_gaussian_vtheta.py`,
  `model_aniso_gaussian_vtheta.py`) and now for the whole structured
  quadratic family too (`model_structured_vtheta.py`: `QuadraticWellVTheta`
  / `LowRankQuadraticVTheta` / `MixtureQuadraticVTheta` (SQ3) /
  `HybridQuadraticVTheta`, the last of which reports the quadratic
  backbone only — its MLP residual has no closed form, so its reading is
  a lower bound, not a ceiling). Only the plain MLP $V_\theta$ baseline
  has no closed-form curvature and is skipped with a clear message per
  checkpoint rather than a crash.

This does **not** require the checkpoint to have been trained with the
CfC/BAOAB integrator: the probe briefly forces the harmonic linearisation
on for one forward pass regardless of what the model actually trained
with, reads off $K$, then restores the checkpoint's own config. Set
`STIFFNESS_CKPT_PATHS` in Cell 0 to a list of paths and re-run this phase;
leave it empty (the default) to skip.

In [ ]:
# ── Cell 12: Phase 7 — Stiffness Audit ────────────────────────────

print('=' * 64)
print('  PHASE 7: Stiffness Audit')
print('=' * 64)

if not STIFFNESS_CKPT_PATHS:
    print('Skipped (STIFFNESS_CKPT_PATHS is empty).')
else:
    import dataclasses as _dc7

    def _build_model_from_ckpt_cfg(path):
        """Rebuild a model from a checkpoint that carries 'model_cfg'.

        Deliberately narrower than Cell 4: this only supports the
        ``from_cfg`` path (every checkpoint the production OWT notebooks
        save has one), since a stiffness audit is meaningless unless the
        rebuilt model is provably the one whose weights are being scored.
        """
        _ckpt = torch.load(path, map_location='cpu', weights_only=False)
        if 'model_cfg' not in _ckpt:
            raise RuntimeError(
                f'{path}: no model_cfg in checkpoint. Phase 7 only supports '
                'checkpoints saved by the production OWT notebooks (they '
                "always embed 'model_cfg' and 'step').")

        _cfg_dict = dict(_ckpt['model_cfg'])
        _known = {f.name for f in _dc7.fields(FockMultiXiPARFConfig)}
        _unknown = sorted(set(_cfg_dict) - _known)
        if _unknown:
            raise RuntimeError(
                f'{path}: checkpoint config has fields this checkout does '
                f'not know: {_unknown}. Pull semsimula-paper and restart.')

        _logfreq_path = _cfg_dict.get('logfreq_path', '')
        if _logfreq_path and not Path(_logfreq_path).exists():
            _local_lf = CA_DIR / 'scaleup' / 'results' / Path(_logfreq_path).name
            if _local_lf.exists():
                _cfg_dict['logfreq_path'] = str(_local_lf)
            else:
                raise RuntimeError(
                    f'{path}: logfreq_path {_logfreq_path!r} not found '
                    'locally either -- run Cell 4 once first so it gets '
                    'computed and cached under scaleup/results/.')

        _mcfg = FockMultiXiPARFConfig(**_cfg_dict)
        _mdl = FockMultiXiPARFLM(_mcfg)

        # model_cfg never records that V_theta was swapped from the
        # default MLP to a Gaussian family post-construction (same gap
        # Cell 4 has -- see _detect_vtheta_family_from_state_dict there).
        # Without this, every model_cfg-carrying Gaussian checkpoint would
        # rebuild as an untrained MLP and get silently reported as
        # '[SKIP] ... no harmonic_terms()' below instead of actually
        # being audited.
        _vt_family7, _vt_info7 = _detect_vtheta_family_from_state_dict(
            _ckpt['model_state_dict'])
        if _vt_family7 in ('gaussian_iso', 'gaussian_aniso'):
            _d7, _K7, _n_ctx7 = _vt_info7['d'], _vt_info7['K'], _vt_info7['n_ctx']
            if _vt_family7 == 'gaussian_aniso':
                from model_aniso_gaussian_vtheta import (
                    AnisotropicDepthConditionedGaussianVTheta,
                    install_aniso_depth_routing)
                _mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
                    d=_d7, K=_K7, n_ctx=_n_ctx7, n_layers=_mcfg.L,
                    rank=_vt_info7['rank'], w_scale=1.0,
                    init_log_precision=-_math.log(_d7),
                    precision_max=2.0 / _d7, code_init_std=0.02,
                )
                install_aniso_depth_routing(_mdl)
            else:
                from model_gaussian_vtheta import (
                    DepthConditionedMultiContextGaussianVTheta,
                    install_depth_routing)
                _mdl.V_theta = DepthConditionedMultiContextGaussianVTheta(
                    d=_d7, K=_K7, n_ctx=_n_ctx7, n_layers=_mcfg.L,
                    w_scale=1.0, init_log_precision=-_math.log(_d7),
                    precision_max=2.0 / _d7, code_init_std=0.02,
                )
                install_depth_routing(_mdl)
            print(f'  V_theta family (from state_dict): {_vt_family7}  '
                  f'{_vt_info7}')
        elif _vt_family7 not in ('mlp',):
            print(f'  [WARN] {Path(path).name}: V_theta family detected as '
                  f'{_vt_family7!r} -- no from_cfg swap-in for this family; '
                  'leaving the default MLP V_theta (check missing/'
                  'unexpected below).')

        _missing, _unexpected = _mdl.load_state_dict(
            _ckpt['model_state_dict'], strict=False)
        if _missing or _unexpected:
            print(f'  [WARN] {Path(path).name}: missing={_missing} '
                  f'unexpected={_unexpected}')
        _mdl.to(DEVICE).eval()

        _step = _ckpt.get('step')
        _val_ppl = _ckpt.get('val_ppl')
        del _ckpt
        return _mdl, _mcfg, _step, _val_ppl

    class _StructuredVThetaMultiXiAdapter(nn.Module):
        """(xis: (B,T,K,d), h: (B,T,d)) -> V: (B,T,1).

        Same interface as the inline adapter class defined in
        colab_fock_multixi_structured_vtheta.ipynb's Cell 12, plus a
        harmonic_terms() passthrough so Phase 7 can score these
        checkpoints (that notebook's own copy has no such passthrough
        today -- only forward/analytical_grad/attractor_centres).
        """

        def __init__(self, inner, K, d):
            super().__init__()
            self.inner = inner
            self.K = K
            self.d = d

        def forward(self, xis, h):
            B, T, K, d = xis.shape
            return self.inner(xis.reshape(B, T, K * d), h)

        def analytical_grad(self, xis, h):
            B, T, K, d = xis.shape
            return self.inner.analytical_grad(xis.reshape(B, T, K * d), h)

        def harmonic_terms(self, xis, h, *, comps=None):
            # comps is accepted (not used) purely for call-signature
            # compatibility with _layer_step_langevin, which always
            # passes it as a keyword regardless of V_theta family; the
            # structured quadratic classes have no cached-components
            # path (no context_components()), so it is always None here.
            if not hasattr(self.inner, 'harmonic_terms'):
                raise AttributeError(
                    f'{type(self.inner).__name__} has no harmonic_terms()')
            B, T, K, d = xis.shape
            return self.inner.harmonic_terms(xis.reshape(B, T, K * d), h)

    def _build_model_from_recipe_ckpt(path):
        """Rebuild a model from a checkpoint that carries a 'recipe' dict
        instead of a full 'model_cfg' -- the schema
        colab_fock_multixi_structured_vtheta.ipynb's checkpoints use (SQ1-4
        structured V_theta recipes + the MLP baseline, on TinyStories).

        Mirrors that notebook's shared-architecture cell and its per-recipe
        V_theta swap-in exactly (D=256, L=8, V_HIDDEN=1024, ...): there is
        no embedded config in these checkpoints to read the architecture
        back from, so if that notebook's shared-architecture cell changes,
        this needs updating too.
        """
        _ckpt = torch.load(path, map_location='cpu', weights_only=False)
        if 'model_cfg' in _ckpt:
            raise RuntimeError(f'{path}: has model_cfg -- use '
                                '_build_model_from_ckpt_cfg instead.')
        if 'recipe' not in _ckpt:
            raise RuntimeError(
                f"{path}: no 'recipe' (and no 'model_cfg') in checkpoint. "
                'Phase 7 only supports the two checkpoint schemas the '
                'production notebooks save.')

        _recipe = _ckpt['recipe']
        _vkind = _recipe['v_theta_kind']
        _xi_channels = _recipe['xi_channels']

        _D, _L = 256, 8
        _V_HIDDEN, _V_DEPTH = 1024, 3
        _VOCAB_SIZE, _MAX_LEN = 50257, 1024
        _DT = 1.0
        _fixed_gamma = float(_ckpt.get('gamma', 0.30))

        if _xi_channels == 4:
            _xi_alpha_inits = [0.25, 0.50, 0.75, 0.95]
        elif _xi_channels == 8:
            _xi_alpha_inits = [0.05, 0.15, 0.30, 0.50, 0.70, 0.85, 0.93, 0.98]
        else:
            raise RuntimeError(f'{path}: no default alpha inits for '
                                f'xi_channels={_xi_channels}')

        _lf_dir = CA_DIR / 'scaleup' / 'results'
        _lf_file = _lf_dir / f'logfreq_surprisal_{DATASET}.npy'
        if not _lf_file.exists():
            _lf_dir.mkdir(parents=True, exist_ok=True)
            _counts = np.bincount(
                train_ids.astype(np.int64), minlength=_VOCAB_SIZE,
            ).astype(np.float64)
            _p = (_counts + 1.0) / (_counts.sum() + _VOCAB_SIZE)
            np.save(str(_lf_file), (-np.log(_p)).astype(np.float32))

        _mcfg = FockMultiXiPARFConfig(
            vocab_size=_VOCAB_SIZE, d=_D, max_len=_MAX_LEN, L=_L,
            v_hidden=_V_HIDDEN, v_depth=_V_DEPTH, dt=_DT,
            mass_mode='logfreq', logfreq_path=str(_lf_file),
            logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=_fixed_gamma,
            causal_force=True, ln_after_step=True,
            xi_channels=_xi_channels, xi_alpha_inits=_xi_alpha_inits,
            xi_learnable=True, xi_alpha_init_mode='explicit',
            v_phi_kind='structural_competitive',
            v_phi_phi_hidden=128, v_phi_theta_hidden=128,
            top_k=8, score_head_hidden=32,
            gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
            use_gathered_v_phi=True, use_layer_checkpoint=True,
            ln_before_distance=True, per_layer_v_phi_scale=True,
            fock_version='v2', n_registers=16,
            register_salience_decay=0.5, register_salience_threshold=0.005,
            creation_gate_hidden=64, stack_discipline=True,
            d_k=64, tau_create_init=8.0, reverse_channel=True,
            per_register_tau=True, per_register_keys=True,
            ortho_register_init=True, prefix_causal_registers=True,
        )
        _mdl = FockMultiXiPARFLM(_mcfg)
        _xi_d = _xi_channels * _D

        from model_structured_vtheta import (
            MixtureQuadraticVTheta, QuadraticWellVTheta,
            LowRankQuadraticVTheta, HybridQuadraticVTheta,
        )

        if _vkind == 'sq3':
            _inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
            nn.Module.__init__(_inner)
            _inner.d = _D
            _inner.K = _recipe['K_mix']
            _inner.tau = _recipe['tau']
            _inner.mu_proj = nn.Linear(_xi_d, _recipe['K_mix'] * _D)
            _inner.a_proj = nn.Linear(_xi_d, _recipe['K_mix'] * _D)
            _inner.pi_proj = nn.Linear(_xi_d, _recipe['K_mix'])
            _inner.b_proj = nn.Linear(_xi_d, 1)
            _inner._init_weights(0.0)
            _mdl.V_theta = _StructuredVThetaMultiXiAdapter(_inner, K=_xi_channels, d=_D)
        elif _vkind == 'sq1':
            _inner = QuadraticWellVTheta.__new__(QuadraticWellVTheta)
            nn.Module.__init__(_inner)
            _inner.d = _D
            _inner.mu_proj = nn.Linear(_xi_d, _D)
            _inner.a_proj = nn.Linear(_xi_d, _D)
            _inner.b_proj = nn.Linear(_xi_d, 1)
            _inner._init_weights(0.0)
            _mdl.V_theta = _StructuredVThetaMultiXiAdapter(_inner, K=_xi_channels, d=_D)
        elif _vkind == 'sq2':
            _inner = LowRankQuadraticVTheta.__new__(LowRankQuadraticVTheta)
            nn.Module.__init__(_inner)
            _inner.d = _D
            _inner.rank = _recipe['rank']
            _inner.mu_proj = nn.Linear(_xi_d, _D)
            _inner.lam_proj = nn.Linear(_xi_d, _D)
            _inner.U_proj = nn.Linear(_xi_d, _D * _recipe['rank'])
            _inner.b_proj = nn.Linear(_xi_d, 1)
            _inner._init_weights(0.0)
            _mdl.V_theta = _StructuredVThetaMultiXiAdapter(_inner, K=_xi_channels, d=_D)
        elif _vkind == 'sq4':
            _inner = HybridQuadraticVTheta.__new__(HybridQuadraticVTheta)
            nn.Module.__init__(_inner)
            _inner.d = _D
            _inner.quad = QuadraticWellVTheta.__new__(QuadraticWellVTheta)
            nn.Module.__init__(_inner.quad)
            _inner.quad.d = _D
            _inner.quad.mu_proj = nn.Linear(_xi_d, _D)
            _inner.quad.a_proj = nn.Linear(_xi_d, _D)
            _inner.quad.b_proj = nn.Linear(_xi_d, 1)
            _inner.quad._init_weights(0.0)
            _h_h, _h_d = _recipe['hybrid_v_hidden'], _recipe['hybrid_v_depth']
            _layers = [nn.Linear(_xi_d + _D, _h_h), nn.GELU()]
            for _ in range(_h_d - 1):
                _layers += [nn.Linear(_h_h, _h_h), nn.GELU()]
            _layers += [nn.Linear(_h_h, 1)]
            _inner.mlp = nn.Sequential(*_layers)
            _inner.alpha = nn.Parameter(torch.tensor(0.1))
            _mdl.V_theta = _StructuredVThetaMultiXiAdapter(_inner, K=_xi_channels, d=_D)
        elif _vkind == 'mlp':
            pass  # keep the default MLP V_theta FockMultiXiPARFLM was built with
        else:
            raise RuntimeError(f'{path}: unknown v_theta_kind={_vkind!r}')

        _missing, _unexpected = _mdl.load_state_dict(
            _ckpt['model_state_dict'], strict=False)
        if _missing or _unexpected:
            print(f'  [WARN] {Path(path).name}: missing={_missing} '
                  f'unexpected={_unexpected}')
        _mdl.to(DEVICE).eval()

        _step = _ckpt.get('step')
        _val_ppl = _ckpt.get('val_ppl')
        del _ckpt
        return _mdl, _mcfg, _step, _val_ppl

    def _stiffness_report(mdl, batches, dt=None):
        """omega*dt distribution over layers/tokens/dims, several batches.

        Forces the harmonic linearisation on for the duration of these
        forward passes regardless of the checkpoint's own integrator, then
        restores it -- this is a read-only probe, not a re-training run.
        Returns None (with a printed reason) if V_theta has no closed-form
        curvature.
        """
        if not hasattr(mdl.V_theta, 'harmonic_terms'):
            print(f'  [SKIP] V_theta={type(mdl.V_theta).__name__} has no '
                  'harmonic_terms() (plain MLP V_theta baseline -- no '
                  'closed-form curvature).')
            return None

        dt = float(mdl.cfg.dt if dt is None else dt)
        seen = []
        _orig = mdl.V_theta.harmonic_terms

        def _recording(xis, h, comps=None):
            # _layer_step_langevin always calls harmonic_terms(xis, h,
            # comps=...) -- comps must be accepted and forwarded here
            # even though this probe never precomputes it itself.
            k_diag, s = _orig(xis, h, comps=comps)
            seen.append(k_diag.detach().float().flatten().cpu())
            return k_diag, s

        _saved = (mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force)
        mdl.V_theta.harmonic_terms = _recording
        mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = 'baoab_cfc', True
        try:
            with torch.enable_grad():
                for x in batches:
                    mdl(x)
        finally:
            mdl.V_theta.harmonic_terms = _orig
            mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = _saved

        k = torch.cat(seen)
        m = float(mdl.compute_mass(batches[0]).mean())
        wdt = (k.clamp(min=0) / m).sqrt() * dt
        n_total = int(wdt.numel())
        wdt_max = float(wdt.max())
        if n_total > 4_000_000:
            idx = torch.randint(0, n_total, (4_000_000,))
            wdt_q = wdt[idx]
        else:
            wdt_q = wdt
        q = torch.tensor([0.5, 0.9, 0.99, 0.999])
        qs = torch.quantile(wdt_q.double(), q.double()).float()
        return {
            'n_samples': n_total, 'mean_mass': m,
            'median': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
            'p999': float(qs[3]), 'max': wdt_max,
            'frac_marginal': float((wdt > 1.0).float().mean()),
            'frac_unstable': float((wdt > 2.0).float().mean()),
        }

    _rng7 = np.random.default_rng(0)
    _stiff_rows = []
    for _path in STIFFNESS_CKPT_PATHS:
        print(f'\n--- {Path(_path).name} ---')
        try:
            _mdl7, _mcfg7, _step7, _ppl7 = _build_model_from_ckpt_cfg(_path)
        except RuntimeError as e1:
            if 'no model_cfg in checkpoint' not in str(e1):
                print(f'  [ERROR] {e1}')
                continue
            # Falls back to the 'recipe'-carrying schema (e.g.
            # colab_fock_multixi_structured_vtheta.ipynb's SQ1-4 / MLP runs)
            # rather than treating this as a hard failure.
            try:
                _mdl7, _mcfg7, _step7, _ppl7 = _build_model_from_recipe_ckpt(_path)
            except Exception as e2:
                print(f'  [ERROR] {e2}')
                continue
        except Exception as e1:
            print(f'  [ERROR] {e1}')
            continue

        _batches7 = []
        for _ in range(STIFFNESS_N_BATCHES):
            _xb7, _ = get_batch(val_ids, DIAG_BATCH_SZ, BLOCK_SIZE, _rng7)
            _batches7.append(torch.from_numpy(_xb7).to(DEVICE))

        _rep7 = _stiffness_report(_mdl7, _batches7)
        if _rep7 is not None:
            _rep7['path'] = str(_path)
            _rep7['step'] = _step7
            _rep7['val_ppl'] = _ppl7
            _stiff_rows.append(_rep7)
            print(f'  step={_step7}  val_ppl={_ppl7}')
            print(f'  omega*dt: median={_rep7["median"]:.4f}  '
                  f'p90={_rep7["p90"]:.4f}  p99={_rep7["p99"]:.4f}  '
                  f'p99.9={_rep7["p999"]:.4f}  max={_rep7["max"]:.4f}')
            print(f'  frac(omega*dt>1)={_rep7["frac_marginal"]:.3e}  '
                  f'frac(omega*dt>2, Verlet-unstable)={_rep7["frac_unstable"]:.3e}')

        del _mdl7, _mcfg7, _batches7
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if _stiff_rows:
        _stiff_rows.sort(key=lambda r: (r['step'] is None, r['step']))
        print(f'\n{"step":>8}  {"val_ppl":>9}  {"median":>7}  {"p90":>7}  '
              f'{"p99":>7}  {"p99.9":>7}  {"max":>9}  {"frac>2":>9}')
        print('-' * 76)
        for r in _stiff_rows:
            print(f'{str(r["step"]):>8}  {str(r["val_ppl"]):>9}  '
                  f'{r["median"]:7.4f}  {r["p90"]:7.4f}  {r["p99"]:7.4f}  '
                  f'{r["p999"]:7.4f}  {r["max"]:9.4f}  {r["frac_unstable"]:9.3e}')

        stiff_path = OUT_PATH / 'phase7_stiffness_audit.json'
        with open(str(stiff_path), 'w') as f:
            json.dump(_stiff_rows, f, indent=2, default=str)
        print(f'\nSaved: {stiff_path}')

        _steps_ok = [r['step'] for r in _stiff_rows if r['step'] is not None]
        if len(_steps_ok) >= 2:
            fig7, ax7 = plt.subplots(figsize=(9, 5))
            steps7 = [r['step'] for r in _stiff_rows]
            ax7.plot(steps7, [r['p99'] for r in _stiff_rows], 'o-',
                     label='p99', color='#3498db')
            ax7.plot(steps7, [r['max'] for r in _stiff_rows], 's--',
                     label='max', color='#e74c3c')
            ax7.axhline(y=2.0, color='black', linewidth=1.5, linestyle=':',
                        label='Verlet stability bound (omega*dt=2)')
            for ws in WATCHDOG_RELOAD_STEPS:
                ax7.axvline(x=ws, color='#95a5a6', linewidth=1, alpha=0.7)
            ax7.set_xlabel('Training step')
            ax7.set_ylabel('omega * dt')
            ax7.set_title('Stiffness audit: omega*dt vs. training step\n'
                           '(grey lines = logged watchdog reloads)')
            ax7.legend()
            fig7.tight_layout()
            stiff_plot_path = OUT_PATH / 'phase7_stiffness_vs_step.png'
            fig7.savefig(str(stiff_plot_path), dpi=150)
            print(f'Saved: {stiff_plot_path}')
            plt.show()
    else:
        print('\nNo usable reports (all checkpoints skipped or errored).')